# 02_baseline.ipynb

# Baseline CNN for Brain Tumor MRI Classification

Notebook này xây dựng pipeline baseline cho dự án phân loại ảnh MRI não.

## Nội dung chính

1. Import thư viện.
2. Khai báo đường dẫn dữ liệu.
3. Kiểm tra dữ liệu đã chia train/val/test.
4. Tiền xử lý dữ liệu ảnh trước khi đưa vào model.
5. Data augmentation cho tập train.
6. Xây dựng Baseline CNN.
7. Train model.
8. Đánh giá bằng accuracy, precision, recall, F1-score.
9. Vẽ confusion matrix.
10. Lưu model và báo cáo.

## Cấu trúc dữ liệu đầu vào

```text
data/
└── processed/
    ├── train/
    │   ├── glioma/
    │   ├── meningioma/
    │   ├── notumor/
    │   └── pituitary/
    ├── val/
    │   ├── glioma/
    │   ├── meningioma/
    │   ├── notumor/
    │   └── pituitary/
    └── test/
        ├── glioma/
        ├── meningioma/
        ├── notumor/
        └── pituitary/
```

## 1. Import Libraries

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger
)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", len(tf.config.list_physical_devices("GPU")) > 0)

## 2. Configuration

Khai báo đường dẫn và tham số chính.

Notebook này dùng dữ liệu đã chia ở:

```text
data/processed/train
data/processed/val
data/processed/test
```

In [ ]:
PROJECT_ROOT = Path(".")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

TRAIN_DIR = PROCESSED_DATA_DIR / "train"
VAL_DIR = PROCESSED_DATA_DIR / "val"
TEST_DIR = PROCESSED_DATA_DIR / "test"

MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"
LOG_DIR = PROJECT_ROOT / "logs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224, 224)
IMG_HEIGHT = IMG_SIZE[0]
IMG_WIDTH = IMG_SIZE[1]
CHANNELS = 3

BATCH_SIZE = 32
EPOCHS = 20
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Project root        : {PROJECT_ROOT.resolve()}")
print(f"Processed data path : {PROCESSED_DATA_DIR}")
print(f"Train path          : {TRAIN_DIR}")
print(f"Validation path     : {VAL_DIR}")
print(f"Test path           : {TEST_DIR}")

## 3. Check Dataset Folders

In [ ]:
required_dirs = {
    "train": TRAIN_DIR,
    "validation": VAL_DIR,
    "test": TEST_DIR
}

for split_name, dir_path in required_dirs.items():
    if not dir_path.exists():
        raise FileNotFoundError(f"Không tìm thấy folder {split_name}: {dir_path}")

print("Tất cả folder train/val/test đã tồn tại.")

## 4. Count Images Per Class

Bước này giúp kiểm tra số lượng ảnh trong từng class của từng tập dữ liệu.

In [ ]:
def count_images_by_class(directory):
    image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]
    records = []

    for class_dir in sorted(directory.iterdir()):
        if class_dir.is_dir():
            image_count = 0

            for ext in image_extensions:
                image_count += len(list(class_dir.glob(ext)))

            records.append({
                "class_name": class_dir.name,
                "image_count": image_count
            })

    return pd.DataFrame(records)


train_count_df = count_images_by_class(TRAIN_DIR)
val_count_df = count_images_by_class(VAL_DIR)
test_count_df = count_images_by_class(TEST_DIR)

print("TRAIN DATASET")
display(train_count_df)

print("VALIDATION DATASET")
display(val_count_df)

print("TEST DATASET")
display(test_count_df)

print("Total train images      :", train_count_df["image_count"].sum())
print("Total validation images :", val_count_df["image_count"].sum())
print("Total test images       :", test_count_df["image_count"].sum())

## 5. Visualize Dataset Distribution

In [ ]:
def plot_class_distribution(df, title, save_path):
    plt.figure(figsize=(8, 5))
    plt.bar(df["class_name"], df["image_count"])
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()


plot_class_distribution(
    train_count_df,
    "Train Dataset Distribution",
    FIGURE_DIR / "train_distribution.png"
)

plot_class_distribution(
    val_count_df,
    "Validation Dataset Distribution",
    FIGURE_DIR / "val_distribution.png"
)

plot_class_distribution(
    test_count_df,
    "Test Dataset Distribution",
    FIGURE_DIR / "test_distribution.png"
)

## 6. Tiền xử lý dữ liệu trước khi đưa vào model

Các bước tiền xử lý:

### 6.1 Resize ảnh

Toàn bộ ảnh được resize về:

```text
224 x 224
```

### 6.2 Chuyển ảnh về RGB

Input của CNN là:

```text
224 x 224 x 3
```

### 6.3 Normalize pixel

Pixel ảnh từ khoảng:

```text
0 - 255
```

được đưa về:

```text
0 - 1
```

bằng `rescale=1.0/255.0`.

### 6.4 Convert label

Keras tự động chuyển tên folder thành nhãn số.

### 6.5 Data augmentation

Chỉ áp dụng cho tập train để giảm overfitting.

## 7. Create Image Data Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0
)

train_data = train_datagen.flow_from_directory(
    directory=TRAIN_DIR,
    target_size=IMG_SIZE,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_data = val_test_datagen.flow_from_directory(
    directory=VAL_DIR,
    target_size=IMG_SIZE,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_data = val_test_datagen.flow_from_directory(
    directory=TEST_DIR,
    target_size=IMG_SIZE,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

NUM_CLASSES = train_data.num_classes
CLASS_NAMES = list(train_data.class_indices.keys())

print("Number of classes:", NUM_CLASSES)
print("Class names:", CLASS_NAMES)
print("Class indices:", train_data.class_indices)

## 8. Visualize Preprocessed Images

In [ ]:
images, labels = next(train_data)

plt.figure(figsize=(10, 10))

for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])

    class_index = np.argmax(labels[i])
    class_name = CLASS_NAMES[class_index]

    plt.title(class_name)
    plt.axis("off")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "preprocessed_sample_images.png")
plt.show()

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Pixel min:", images.min())
print("Pixel max:", images.max())

## 9. Build Baseline CNN Model

In [ ]:
def build_baseline_cnn(input_shape, num_classes):
    model = models.Sequential(name="Baseline_CNN")

    model.add(layers.Input(shape=input_shape))

    model.add(layers.Conv2D(32, (3, 3), activation="relu", padding="same"))
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(layers.Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(layers.Conv2D(128, (3, 3), activation="relu", padding="same"))
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(layers.Conv2D(256, (3, 3), activation="relu", padding="same"))
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(layers.Flatten())

    model.add(layers.Dense(256, activation="relu"))
    model.add(layers.Dropout(0.5))

    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dropout(0.3))

    model.add(layers.Dense(num_classes, activation="softmax"))

    return model


input_shape = (IMG_HEIGHT, IMG_WIDTH, CHANNELS)

model = build_baseline_cnn(
    input_shape=input_shape,
    num_classes=NUM_CLASSES
)

model.summary()

## 10. Compile Model

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

## 11. Define Callbacks

In [ ]:
best_model_path = MODEL_DIR / "baseline_cnn_best.keras"
final_model_path = MODEL_DIR / "baseline_cnn_final.keras"
train_log_path = LOG_DIR / "baseline_training_log.csv"

callbacks = [
    ModelCheckpoint(
        filepath=best_model_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    CSVLogger(
        filename=train_log_path,
        append=False
    )
]

## 12. Train Baseline Model

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 13. Plot Training Curves

In [ ]:
def plot_training_history(history):
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(8, 6))
    plt.plot(epochs_range, acc, label="Train Accuracy")
    plt.plot(epochs_range, val_acc, label="Validation Accuracy")
    plt.title("Baseline CNN - Accuracy Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "baseline_accuracy_curve.png")
    plt.show()

    plt.figure(figsize=(8, 6))
    plt.plot(epochs_range, loss, label="Train Loss")
    plt.plot(epochs_range, val_loss, label="Validation Loss")
    plt.title("Baseline CNN - Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "baseline_loss_curve.png")
    plt.show()


plot_training_history(history)

## 14. Evaluate Model on Test Set

In [ ]:
test_loss, test_accuracy = model.evaluate(test_data, verbose=1)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")

## 15. Predict Test Set

In [ ]:
test_data.reset()

y_pred_prob = model.predict(test_data, verbose=1)

y_pred = np.argmax(y_pred_prob, axis=1)

y_true = test_data.classes

print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)

## 16. Classification Metrics

In [ ]:
accuracy = accuracy_score(y_true, y_pred)

precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall_weighted = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

metrics_summary = {
    "model_name": "Baseline CNN",
    "image_size": str(IMG_SIZE),
    "batch_size": BATCH_SIZE,
    "epochs": len(history.history["loss"]),
    "test_loss": float(test_loss),
    "test_accuracy": float(accuracy),
    "precision_macro": float(precision_macro),
    "recall_macro": float(recall_macro),
    "f1_macro": float(f1_macro),
    "precision_weighted": float(precision_weighted),
    "recall_weighted": float(recall_weighted),
    "f1_weighted": float(f1_weighted)
}

metrics_df = pd.DataFrame([metrics_summary])

display(metrics_df)

metrics_df.to_csv(
    REPORT_DIR / "baseline_metrics_summary.csv",
    index=False
)

## 17. Classification Report

In [ ]:
report = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
)

print(report)

report_path = REPORT_DIR / "baseline_classification_report.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print(f"Classification report saved to: {report_path}")

## 18. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Baseline CNN - Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(CLASS_NAMES))
plt.xticks(tick_marks, CLASS_NAMES, rotation=45)
plt.yticks(tick_marks, CLASS_NAMES)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j,
            i,
            cm[i, j],
            horizontalalignment="center",
            verticalalignment="center"
        )

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "baseline_confusion_matrix.png")
plt.show()

## 19. Save Final Model

In [ ]:
model.save(final_model_path)

print(f"Best model saved to : {best_model_path}")
print(f"Final model saved to: {final_model_path}")

## 20. Save Class Indices

In [ ]:
class_indices_path = REPORT_DIR / "class_indices.json"

with open(class_indices_path, "w", encoding="utf-8") as f:
    json.dump(train_data.class_indices, f, ensure_ascii=False, indent=4)

print(f"Class indices saved to: {class_indices_path}")
print(train_data.class_indices)

## 21. Load Best Model and Test Again

In [ ]:
best_model = tf.keras.models.load_model(best_model_path)

best_test_loss, best_test_accuracy = best_model.evaluate(test_data, verbose=1)

print(f"Best Model Test Loss     : {best_test_loss:.4f}")
print(f"Best Model Test Accuracy : {best_test_accuracy:.4f}")

## 22. Predict One Batch

In [ ]:
test_data.reset()

sample_images, sample_labels = next(test_data)

sample_pred_prob = best_model.predict(sample_images)
sample_pred = np.argmax(sample_pred_prob, axis=1)
sample_true = np.argmax(sample_labels, axis=1)

plt.figure(figsize=(12, 12))

for i in range(min(9, len(sample_images))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(sample_images[i])

    true_label = CLASS_NAMES[sample_true[i]]
    pred_label = CLASS_NAMES[sample_pred[i]]

    plt.title(f"True: {true_label}\nPred: {pred_label}")
    plt.axis("off")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "baseline_prediction_samples.png")
plt.show()

## 23. Summary

Notebook `02_baseline.ipynb` đã hoàn thành:

1. Load dữ liệu từ `data/processed`.
2. Kiểm tra dữ liệu train/val/test.
3. Tiền xử lý ảnh:
   - Resize ảnh về `224x224`.
   - Chuyển ảnh về RGB.
   - Normalize pixel về `[0, 1]`.
   - Convert label theo folder.
   - Data augmentation cho tập train.
4. Xây dựng Baseline CNN.
5. Train model.
6. Đánh giá model.
7. Lưu model và báo cáo.

## Output

```text
models/
├── baseline_cnn_best.keras
└── baseline_cnn_final.keras

reports/
├── baseline_metrics_summary.csv
├── baseline_classification_report.txt
├── class_indices.json
└── figures/
    ├── train_distribution.png
    ├── val_distribution.png
    ├── test_distribution.png
    ├── preprocessed_sample_images.png
    ├── baseline_accuracy_curve.png
    ├── baseline_loss_curve.png
    ├── baseline_confusion_matrix.png
    └── baseline_prediction_samples.png

logs/
└── baseline_training_log.csv
```